# ch07 练习 —— 筛选、索引与条件逻辑

数据：`data/device_defects.csv`（270 行 x 13 列）

**每题都有 `assert` 验收块**。跑通即正确，跑不通的报错信息本身就说清了哪里错。
**不要翻答案版。**

| 题号 | 考什么 |
|---|---|
| TODO(1) | 单条件掩码 |
| TODO(2) | 多条件 `&`（括号不能省） |
| TODO(3) | `isin` |
| TODO(4) | `between` |
| TODO(5) | `~` 取反与 `NaN` 的关系 |
| TODO(6) | `query` 多条件 |
| TODO(7) | `query` 引用局部变量（`@`） |
| TODO(8) | `query` + `.str` 方法调用（并澄清"必须换引擎"的旧说法） |
| TODO(9) | `filter` 筛列（不是筛行） |
| TODO(10) | **`where` 与 `mask` 的 `NaN` 差异** |
| TODO(11) | `loc` 条件赋值 |
| TODO(12) | `np.select` 多档分类 |
| 综合 ① | 三条件组合清单 + 图表 |
| 综合 ② | 条件筛选自查报告 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False

DATA = Path("data")
dev = pd.read_csv(DATA / "device_defects.csv")

SENTINEL_CUT = 1000      # 负荷值 >= 该值视为哨兵/尖峰
REPLACE_WITH = 888       # 演示 where / mask 的替换值
RISK_CHOICES = ["高", "中"]

print("pandas", pd.__version__)
print("dev", dev.shape)

---

## 基础题

In [ ]:
crit_mask = dev["缺陷等级"] == "危急"
# ---- 验收 ----
assert crit_mask.dtype == bool, f"掩码 dtype 应为 bool，实际 {crit_mask.dtype}"
assert int(crit_mask.sum()) == 31, f"危急应有 31 条，实际 {int(crit_mask.sum())}"
assert len(crit_mask) == 270, "掩码长度必须等于行数"
print("TODO(1) 通过 -> 危急", int(crit_mask.sum()), "条")

In [ ]:
crit_trans_mask = (dev["缺陷等级"] == "危急") & (dev["设备类型"] == "变压器")
# ---- 验收 ----
assert int(crit_trans_mask.sum()) == 5, f"危急且变压器应有 5 条，实际 {int(crit_trans_mask.sum())}"
# 证明括号不能省：只给一半加括号 -> 静默得到 0 命中（不报错）
half = (dev["缺陷等级"] == "危急") & dev["设备类型"] == "变压器"
assert int(half.sum()) == 0, f"半括号写法应静默得到 0 命中，实际 {int(half.sum())}"
assert int(crit_trans_mask.sum()) == 5, "正确写法仍应是 5"
print("   半括号写法命中 0 条且不报错 -> 这就是为什么要给每个条件都加括号")
print("TODO(2) 通过 -> 危急且变压器", int(crit_trans_mask.sum()), "条")

In [ ]:
sev_crit_mask = dev["缺陷等级"].isin(["严重", "危急"])
# ---- 验收 ----
assert int(sev_crit_mask.sum()) == 106, f"应为 106 条，实际 {int(sev_crit_mask.sum())}"
assert sev_crit_mask.dtype == bool
assert sev_crit_mask.equals(dev["缺陷等级"].isin({"严重", "危急"})), "list 与 set 结果应一致"
print("TODO(3) 通过 -> 严重或危急", int(sev_crit_mask.sum()), "条")

In [ ]:
load_mid_mask = dev["负荷值"].between(100, 300)
# ---- 验收 ----
assert int(load_mid_mask.sum()) == 56, f"应为 56 条，实际 {int(load_mid_mask.sum())}"
manual = (dev["负荷值"] >= 100) & (dev["负荷值"] <= 300)
assert bool((load_mid_mask == manual).all()), "between 应等价于 (>=下限) & (<=上限)"
assert load_mid_mask.dtype == bool, "between 返回纯 bool，不传播 NaN"
print("TODO(4) 通过 -> 负荷值在 [100,300] 的有", int(load_mid_mask.sum()), "条")

In [ ]:
not_normal_mask = ~(dev["缺陷等级"] == "一般")
# ---- 验收 ----
assert int(not_normal_mask.sum()) == 106, f"非一般应为 106（270 - 一般 164），实际 {int(not_normal_mask.sum())}"
strict = (dev["缺陷等级"] != "一般") & dev["缺陷等级"].notna()
assert int(strict.sum()) == int(not_normal_mask.sum()), "本章无缺失时两者应完全相等"
assert dev["缺陷等级"].value_counts().to_dict() == {"一般": 164, "严重": 75, "危急": 31},     "先核对三类基准数"
missing_lvl = int(dev["缺陷等级"].isna().sum())
assert missing_lvl == 0, "本章数据里缺陷等级没有缺失，取反与严格版必然相等"
assert bool((not_normal_mask == strict).all())
print("TODO(5) 通过 -> 非一般", int(not_normal_mask.sum()), "条（本章无缺失，取反版与严格版一致）")

---

## 进阶题

In [ ]:
q_crit_high = dev.query("负荷值 > 500 and 缺陷等级 == '危急'")
# ---- 验收 ----
assert q_crit_high.shape[0] == 5, f"应为 5 行，实际 {q_crit_high.shape[0]}"
assert q_crit_high.shape[1] == 13, "query 不改变列数"
manual = dev[(dev["负荷值"] > 500) & (dev["缺陷等级"] == "危急")]
assert q_crit_high.index.equals(manual.index), "query 与掩码结果应完全一致"
print("TODO(6) 通过 -> query 命中", q_crit_high.shape[0], "行")

In [ ]:
THR = 500

q_var = dev.query("负荷值 > @THR")
# ---- 验收 ----
assert q_var.shape[0] == 45, f"应为 45 行，实际 {q_var.shape[0]}"
assert q_var["负荷值"].min() > THR, "结果里不该有小于等于阈值的行"
try:
    dev.query("负荷值 > THR")
    raise AssertionError("不加 @ 应该报 UndefinedVariableError")
except AssertionError:
    raise
except Exception:
    pass
print("TODO(7) 通过 -> @ 引用局部变量命中", q_var.shape[0], "行")

In [ ]:
q_station_a = dev.query("台区编号.str.startswith('STATION_A')")
# ---- 验收 ----
assert q_station_a.shape[0] == 44, f"应为 44 行，实际 {q_station_a.shape[0]}"
assert q_station_a.shape[1] == 13, "query 不改变列数"
assert q_station_a["台区编号"].str.startswith("STATION_A").all()
manual = dev[dev["台区编号"].str.startswith("STATION_A")]
assert q_station_a.index.equals(manual.index), "query 与掩码结果应完全一致"

# 澄清一个流传很广但已失效的说法：.str 是否必须换 engine="python"？
q_python = dev.query("台区编号.str.startswith('STATION_A')", engine="python")
assert q_python.index.equals(q_station_a.index), "两种引擎结果一致 -> 3.0 默认引擎已支持 .str"
print("TODO(8) 通过 -> 命中", q_station_a.shape[0], "行；默认引擎与 engine='python' 结果一致")

In [ ]:
dev_columns = dev.filter(like="设备")
# ---- 验收 ----
assert list(dev_columns.columns) == ["设备类型", "设备编号"], f"列不对：{list(dev_columns.columns)}"
assert len(dev_columns) == 270, "filter 只筛列，行数一行不少"
assert dev_columns.shape == (270, 2), f"形状应为 (270, 2)，实际 {dev_columns.shape}"

# 对照组：想筛出「变压器」的行，必须用掩码而不是 filter
rows_by_mask = dev[dev["设备类型"] == "变压器"]
assert len(rows_by_mask) == 55, "掩码才是筛行的正确工具"
print("TODO(9) 通过 -> filter 得到", list(dev_columns.columns), "，行数仍为", len(dev_columns))

In [ ]:
CUT = 1000
FILL = 888

w_filled = dev["负荷值"].where(dev["负荷值"] < CUT, FILL)
m_filled = dev["负荷值"].mask(dev["负荷值"] >= CUT, FILL)
# ---- 验收 ----
n_w = int(w_filled.eq(FILL).sum())
n_m = int(m_filled.eq(FILL).sum())
assert n_w == 28, f"where 命中 FILL 应为 28（7 哨兵 + 21 原有缺失），实际 {n_w}"
assert n_m == 7, f"mask 命中 FILL 应为 7（只换哨兵），实际 {n_m}"
assert n_w - n_m == 21, "差值必须等于原有缺失数"
assert int(w_filled.isna().sum()) == 0, "where 给了 other 后不该再有 NaN"
assert int(m_filled.isna().sum()) == 21, "mask 必须保住原有 21 个缺失"

# 想让 where 也保住缺失：把 notna 写进条件
w_safe = dev["负荷值"].where((dev["负荷值"] < CUT) | dev["负荷值"].isna(), FILL)
assert int(w_safe.eq(FILL).sum()) == n_m, "加上 isna() 条件后 where 应与 mask 一致"
print(f"TODO(10) 通过 -> where 换掉 {n_w} 个（含 21 个缺失），mask 只换 {n_m} 个")

In [ ]:
flagged = dev.copy()
flagged.loc[flagged["负荷值"] > 500, "复核标记"] = "高负荷待复核"
flagged["复核标记"] = flagged["复核标记"].fillna("常规")
# ---- 验收 ----
n_flag = int((flagged["复核标记"] == "高负荷待复核").sum())
n_cond = int((flagged["负荷值"] > 500).sum())
assert n_flag == n_cond == 45, f"条件命中 {n_cond} 与赋值命中 {n_flag} 必须相等且为 45"
assert flagged["复核标记"].isna().sum() == 0, "补默认值后不该有缺失"
assert flagged["复核标记"].value_counts().to_dict() == {"常规": 225, "高负荷待复核": 45}

# 反例：链式赋值一行都改不上
chained = dev.copy()
chained[chained["负荷值"] > 500]["备注"] = "xxx"
assert int((chained["备注"] == "xxx").sum()) == 0, "链式赋值应该静默失效"
print("TODO(11) 通过 -> loc 条件赋值命中", n_flag, "行；链式赋值命中 0 行")

In [ ]:
risk_conds = [dev["缺陷等级"].eq("危急"), dev["缺陷等级"].eq("严重")]
risk = np.select(risk_conds, RISK_CHOICES, default="低")
# ---- 验收 ----
assert isinstance(risk, np.ndarray), f"np.select 返回 ndarray，实际 {type(risk).__name__}"
assert len(risk) == 270, "长度必须等于行数"
counts = pd.Series(risk).value_counts().to_dict()
assert counts == {"低": 164, "中": 75, "高": 31}, f"分布不对：{counts}"
assert pd.Series(risk).isna().sum() == 0, "default 保证不产生缺失"

# 反证：条件顺序写反会出错
reversed_risk = np.select([dev["负荷值"] > 100, dev["负荷值"] > 500], ["中", "高"], default="低")
assert int(pd.Series(reversed_risk).eq("高").sum()) == 0,     "宽条件在前时，窄条件永远命中不了（这就是顺序陷阱）"
print("TODO(12) 通过 -> 风险分布", counts)

---

## 综合题

### 综合 ① 高优先检修清单

从 270 条缺陷记录里筛出需要**优先安排检修**的记录，条件三条（缺一不可）：

1. `缺陷等级` ∈ {严重, 危急}
2. `设备类型` ∈ {变压器, 断路器}
3. `处理状态` 含「处理」二字

产出 `urgent`（保留 `记录ID`/`台区编号`/`设备类型`/`缺陷等级`/`负荷值`/`处理状态` 六列），
并画一张「按缺陷等级 × 设备类型」的清点柱状图。

**这一步的关键不是筛出来，而是证明自己没筛错**：三个条件各自的命中数、
交集数、清单行数必须能对上。

In [ ]:
cond_level = dev["缺陷等级"].isin(["严重", "危急"])
cond_type = dev["设备类型"].isin(["变压器", "断路器"])
cond_status = dev["处理状态"].str.contains("处理", na=False)

urgent = dev.loc[
    cond_level & cond_type & cond_status,
    ["记录ID", "台区编号", "设备类型", "缺陷等级", "负荷值", "处理状态"],
]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
urgent.groupby(["缺陷等级", "设备类型"]).size().unstack(fill_value=0).plot.bar(ax=ax)
ax.set_title("高优先检修清单：缺陷等级 x 设备类型")
ax.set_xlabel("缺陷等级")
ax.set_ylabel("条数")
plt.tight_layout()
plt.show()
# ---- 验收 ----
assert len(urgent) == 38, f"清单应有 38 行，实际 {len(urgent)}"
assert list(urgent.columns) == ["记录ID", "台区编号", "设备类型", "缺陷等级", "负荷值", "处理状态"]
# 三个条件逐一自查
assert int(cond_level.sum()) == 106
assert int(cond_type.sum()) == 96, f"变压器55+断路器41 应为 96，实际 {int(cond_type.sum())}"
assert int(cond_status.sum()) == 257
assert int((cond_level & cond_type & cond_status).sum()) == len(urgent),     "三条件交集数必须等于清单行数"
# 清单内部结构
assert urgent["缺陷等级"].value_counts().to_dict() == {"严重": 30, "危急": 8}
assert urgent["设备类型"].value_counts().to_dict() == {"断路器": 20, "变压器": 18}
# 清单里不能有不符合条件的行（反向抽查）
assert urgent["缺陷等级"].isin(["严重", "危急"]).all()
assert urgent["设备类型"].isin(["变压器", "断路器"]).all()
assert urgent["处理状态"].str.contains("处理", na=False).all()
# 图
assert ax is not None and len(ax.patches) >= 4, f"柱状图柱子数不对：{len(ax.patches)}"
print("综合① 通过 -> 清单", urgent.shape)
print(urgent["缺陷等级"].value_counts().to_string())

### 综合 ② 条件筛选自查报告

写一个自查函数的核心逻辑：给定掩码，输出它的质量指标。
竞赛里"数据清洗"那几分的交出物之一就是**这种能自证的过程记录**。

In [ ]:
mask_audit = crit_mask
n_hit = int(mask_audit.sum())
n_total = len(mask_audit)
hit_ratio = round(n_hit / n_total, 4)
complement_hit = int((~mask_audit).sum())
complement_nan_extra = int((~mask_audit & crit_mask.isna()).sum())
audit = {
    "命中数": n_hit,
    "总行数": n_total,
    "命中占比": hit_ratio,
    "反选命中数": complement_hit,
    "取反额外带入的缺失数": complement_nan_extra,
}
# ---- 验收 ----
expect = {"命中数", "总行数", "命中占比", "反选命中数", "取反额外带入的缺失数"}
assert set(audit) == expect, f"键不对。多 {set(audit) - expect}，缺 {expect - set(audit)}"
assert audit["命中数"] == 31 and audit["总行数"] == 270
assert abs(audit["命中占比"] - 0.1148) < 0.0005, f"占比应约 0.1148，实际 {audit['命中占比']}"
assert audit["反选命中数"] == 239, f"反选应为 239，实际 {audit['反选命中数']}"
assert audit["命中数"] + audit["反选命中数"] == audit["总行数"], "命中 + 反选必须等于总数"
assert audit["取反额外带入的缺失数"] == 0, "本章缺陷等级无缺失，取反不会多带入缺失"
print("综合② 通过 -> 条件自查报告：")
for key, value in audit.items():
    print(f"   {key:20s} = {value!r}")

---

### 复盘提问

**复盘问自己**（五题都能答上来才算过）：

1. `dev[mask]["列"] = v` 为什么不生效？正确写法是什么？怎么自查？
2. `~mask` 和"严格反选"差在哪？什么情况下两者结果不同？
3. `where(cond, 888)` 和 `mask(~cond, 888)` 等价吗？为什么实测差 21 行？
4. `filter(like="设备")` 为什么筛不出一行数据？它到底在筛什么？
5. `np.select` 的条件列表为什么必须"窄条件在前"？给个会翻车的反例。

答不上来的，回 `_notes/错题本.md` 记一笔。